# 01. CTR Exploratory Data Analysis

This notebook summarizes the full training set in Parquet batches and uses a reproducible row sample for plots and numeric summaries. It avoids loading the entire 10 GB dataset into memory.

## Data snapshot findings

The full training data scan found 10,704,179 rows and 119 columns; the test data contains 1,527,298 rows and 119 columns. The target is `clicked`, with 204,179 clicks and 10,500,000 non-clicks, for an overall CTR of 1.907%. This is roughly a 51:1 class imbalance, so accuracy alone is not a useful model-selection metric.

Missing values are present in feature columns but not in the target. The `feat_e_3` column has 1,085,557 missing rows (about 10.1%). `gender`, `age_group`, and several feature groups have missing values on 17,208 rows; `feat_a_1` through `feat_a_18` each have 18,598 missing values. Inspect the missingness patterns before adding missing-value indicators.

The most common `gender` values are `1` and `2`. `age_group` values `7`, `8`, and `6` are most frequent. Inventory traffic is concentrated in a small number of IDs, led by `2`, `36`, `37`, `29`, and `42`. The charts and tables below calculate CTR by category from the complete training data.

In [ ]:
from collections import Counter
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns

sns.set_theme(style='whitegrid')

def find_data_dir():
    configured_dir = os.environ.get('CTR_DATA_DIR')
    if configured_dir:
        candidate = Path(configured_dir).expanduser().resolve()
        if (candidate / 'train.parquet').is_file() and (candidate / 'test.parquet').is_file():
            return candidate
        raise FileNotFoundError(f'CTR_DATA_DIR does not contain train.parquet and test.parquet: {candidate}')
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates = [base / 'data' / 'raw', base / 'data_project' / 'data' / 'raw']
        for candidate in candidates:
            if (candidate / 'train.parquet').is_file() and (candidate / 'test.parquet').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Could not find data/raw/train.parquet and test.parquet.')

DATA_DIR = find_data_dir()
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
print(f'Data directory: {DATA_DIR}')

In [ ]:
train_file = pq.ParquetFile(TRAIN_PATH)
test_file = pq.ParquetFile(TEST_PATH)
train_columns = train_file.schema.names
test_columns = test_file.schema.names
TARGET = 'clicked'
CAT_COLUMNS = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour', 'seq']
FEATURE_COLUMNS = [column for column in train_columns if column != TARGET]
NUM_COLUMNS = [column for column in FEATURE_COLUMNS if column not in CAT_COLUMNS]

assert TARGET in train_columns
assert 'ID' in test_columns
assert test_columns == ['ID'] + FEATURE_COLUMNS

schema_summary = pd.DataFrame(
    {'dtype': [str(train_file.schema_arrow.field(column).type) for column in train_columns]},
    index=train_columns,
)
print(f'Train: {train_file.metadata.num_rows:,} rows x {len(train_columns):,} columns')
print(f'Test: {test_file.metadata.num_rows:,} rows x {len(test_columns):,} columns')
print(f'Features: {len(CAT_COLUMNS)} categorical and {len(NUM_COLUMNS)} numeric')
display(schema_summary.T)

In [ ]:
row_count = 0
missing_counts = Counter()
label_counts = Counter()
group_counts = {column: {} for column in CAT_COLUMNS if column != 'seq'}
sample_frames = []

for batch_number, batch in enumerate(train_file.iter_batches(batch_size=100_000)):
    frame = batch.to_pandas()
    row_count += len(frame)
    missing_counts.update(frame.isna().sum().to_dict())
    label_counts.update(frame[TARGET].value_counts().to_dict())
    for column in group_counts:
        grouped = frame.groupby(column, dropna=False)[TARGET].agg(['count', 'sum'])
        for value, row in grouped.iterrows():
            key = '<NA>' if pd.isna(value) else str(value)
            totals = group_counts[column].setdefault(key, [0, 0])
            totals[0] += int(row['count'])
            totals[1] += int(row['sum'])
    sample_frames.append(frame.sample(n=min(2_000, len(frame)), random_state=2025 + batch_number))

eda_sample = pd.concat(sample_frames, ignore_index=True)
label_summary = pd.DataFrame(
    {'rows': [label_counts.get(0, 0), label_counts.get(1, 0)]},
    index=['not_clicked', 'clicked'],
)
label_summary['share'] = label_summary['rows'] / row_count
print(f'Rows scanned: {row_count:,}')
display(label_summary)

In [ ]:
missing_summary = pd.Series(missing_counts, name='missing_rows').to_frame()
missing_summary['missing_rate'] = missing_summary['missing_rows'] / row_count
missing_summary = missing_summary.query('missing_rows > 0').sort_values('missing_rate', ascending=False)
print(f'Columns with missing values: {len(missing_summary)} / {len(train_columns)}')
display(missing_summary.head(30).style.format({'missing_rate': '{:.2%}'}))

numeric_summary = eda_sample[NUM_COLUMNS].describe(percentiles=[0.01, 0.5, 0.99]).T
display(numeric_summary.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=label_summary.index, y=label_summary['share'], ax=axes[0], color='steelblue')
axes[0].set_title('Click and non-click share')
axes[0].set_ylabel('Share')
sns.histplot(eda_sample[TARGET], discrete=True, stat='probability', ax=axes[1], color='darkorange')
axes[1].set_title('Target distribution in the row sample')
plt.tight_layout()

category_ctr = {}
for column, counts in group_counts.items():
    rows = []
    for value, (count, clicks) in counts.items():
        rows.append({column: value, 'impressions': count, 'clicks': clicks, 'ctr': clicks / count})
    category_ctr[column] = pd.DataFrame(rows).sort_values('impressions', ascending=False)

for column in ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']:
    print(f'{column}: categories by impression count')
    display(category_ctr[column].head(15).style.format({'ctr': '{:.2%}'}))

In [ ]:
hour_ctr = category_ctr['hour'].copy()
hour_ctr['hour_num'] = pd.to_numeric(hour_ctr['hour'], errors='coerce')
hour_ctr = hour_ctr.sort_values('hour_num')
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=hour_ctr, x='hour_num', y='ctr', marker='o', ax=axes[0])
axes[0].set(title='CTR by hour', xlabel='Hour', ylabel='CTR')
inventory_ctr = category_ctr['inventory_id'].nlargest(15, 'impressions')
sns.barplot(data=inventory_ctr, x='inventory_id', y='ctr', ax=axes[1], color='teal')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set(title='CTR for the 15 largest inventory groups', xlabel='Inventory ID', ylabel='CTR')
plt.tight_layout()

correlation = eda_sample[NUM_COLUMNS].corr()
plt.figure(figsize=(14, 11))
sns.heatmap(correlation, cmap='coolwarm', center=0, xticklabels=False, yticklabels=False)
plt.title('Numeric feature correlation in the row sample')
plt.tight_layout()

sequence_lengths = eda_sample['seq'].fillna('').astype(str).str.split(',').str.len()
sequence_lengths = sequence_lengths[eda_sample['seq'].notna()]
plt.figure(figsize=(8, 4))
sns.histplot(sequence_lengths.clip(upper=sequence_lengths.quantile(0.99)), bins=30)
plt.title('Sequence length (sample; clipped at the 99th percentile)')
plt.xlabel('Number of sequence tokens')
plt.tight_layout()

In [ ]:
id_batch = next(test_file.iter_batches(batch_size=5_000, columns=['ID']))
print('Test ID examples:', id_batch.column(0).to_pylist()[:5])
print('sample_submission.csv exists:', (DATA_DIR / 'sample_submission.csv').is_file())
print(f'Overall CTR: {label_counts.get(1, 0) / row_count:.3%}')
print('Because clicks are rare, report ROC-AUC, PR-AUC, and log loss alongside accuracy.')